#Event Hub producer

Sends telemetry to the cohort Event Hub. The data is real - we replay records from the Lab 2
bronze table instead of inventing synthetic events, so the consumer downstream gets something
worth parsing.

The namespace is shared with the whole group, so every message is stamped with `producer` and
we only send a bounded number of events - no endless loop on shared infrastructure.

In [0]:
%pip install azure-eventhub
dbutils.library.restartPython()

In [0]:
import time

from azure.eventhub import EventHubProducerClient, EventData
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "")
dbutils.widgets.text("eventhub_name", "")
dbutils.widgets.text("n_events", "20000")

login         = dbutils.widgets.get("login")
catalog       = dbutils.widgets.get("target_catalog")
secret_scope  = dbutils.widgets.get("secret_scope")
secret_key    = dbutils.widgets.get("secret_key")
eventhub_name = dbutils.widgets.get("eventhub_name")
n_events      = int(dbutils.widgets.get("n_events"))

assert all([login, catalog, secret_scope, secret_key, eventhub_name])

bronze = f"{login}_bronze"
print(f"{login} -> {eventhub_name}, {n_events} events")

## 1. Build the messages

Take N records from bronze, add who sent them and when, and turn each row into one JSON string.
`to_json(struct("*"))` does the whole row in one go, so adding a field upstream needs no change here.

`producer` is not decoration: the hub is shared, so without it the consumer can't tell our events
from anyone else's.

In [0]:
msgs = [r.value for r in (
    spark.table(f"{catalog}.{bronze}.raw_events_bronze")
    .drop("source_file", "ingestion_ts", "load_date", "_rescued_data")
    .limit(n_events)
    .withColumn("producer", F.lit(login))
    .withColumn("sent_ts", F.current_timestamp())
    .select(F.to_json(F.struct("*")).alias("value"))
    .collect()
)]

print(f"{len(msgs)} messages, avg {sum(len(m) for m in msgs) / len(msgs):.0f} bytes")
print(msgs[0])

In [0]:
from databricks.sdk import WorkspaceClient

WorkspaceClient().secrets.put_secret(
    scope="adam151212_scope",
    key="eventhub-cs",
    string_value="<endpoint>",
)

## 2. Send them

The SDK wants events packed into batches - one send call per batch instead of per event, which is
the difference between seconds and minutes. `batch.add()` raises `ValueError` when the batch is
full (1 MB), and that is the signal to ship it and start a new one.

No partition key on purpose: we don't need per-device ordering, and without a key Event Hubs
spreads batches round-robin over both partitions, which the consumer then reads in parallel.

The namespace has 1 Throughput Unit shared by the whole cohort, so throttling is normal here -
the SDK backs off and retries on its own.

In [0]:
conn = dbutils.secrets.get(secret_scope, secret_key)

kwargs = {} if "EntityPath=" in conn else {"eventhub_name": eventhub_name}
producer = EventHubProducerClient.from_connection_string(conn, **kwargs)

sent, batches, t0 = 0, 0, time.time()
with producer:
    batch = producer.create_batch()
    for m in msgs:
        event = EventData(m)
        try:
            batch.add(event)
        except ValueError:
            producer.send_batch(batch)
            sent, batches = sent + len(batch), batches + 1
            batch = producer.create_batch()
            batch.add(event)
    if len(batch):
        producer.send_batch(batch)
        sent, batches = sent + len(batch), batches + 1

took = time.time() - t0
print(f"sent {sent} events in {batches} batches, {took:.1f}s ({sent / took:.0f} events/s)")